### **<h3 style="color:pink;"> RAG System — Week 7: QLoRA Fine-tuning**

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Introduction**</span>

</div>

In Week 6 we prepared 999 legal instruction-answer pairs in Alpaca format.

This week we fine-tune **Mistral-7B** using **QLoRA** (4-bit quantization + LoRA adapters)!

**What we'll do:**
- ✅ Generate the Google Colab fine-tuning notebook
- ✅ Fine-tune Mistral-7B with QLoRA (run on Colab free GPU)
- ✅ Push fine-tuned model to HuggingFace Hub
- ✅ Evaluate fine-tuned model vs baseline with RAGAS
- 🎯 Target faithfulness: **0.840+** (up from 0.6780!)

**Why QLoRA?**
- Mistral-7B is 7 billion parameters — too big for a laptop
- QLoRA = 4-bit quantization (shrinks model ~4x) + LoRA adapters (trains only small matrices)
- Result: fine-tune a 7B model on a **free Colab T4 GPU** in ~2 hours!

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Setup & Imports**</span>

</div>

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import os
import random
import numpy as np
import faiss
import re
import time
import mlflow

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas import evaluate
from datasets import Dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import InferenceClient

random.seed(42)

mlflow.set_tracking_uri("sqlite:///C:/Users/USER/Documents/RAG_Project/mlflow.db")
mlflow.set_experiment("RAG_Legal_Evaluation")

print("✅ All imports successful!")

✅ All imports successful!


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Understanding QLoRA**</span>

</div>

```
┌─────────────────────────────────────────────────────────────────┐
│                    HOW QLoRA WORKS                              │
│                                                                 │
│  Original Mistral-7B (frozen, 4-bit)   LoRA Adapters (trained) │
│  ┌──────────────────────────┐          ┌──────────────────────┐ │
│  │  Attention (Q, K, V, O)  │    +     │  A matrix (r=16)     │ │
│  │  MLP layers              │          │  B matrix (r=16)     │ │
│  │  [~7B params, frozen]    │          │  [~8M params, train] │ │
│  └──────────────────────────┘          └──────────────────────┘ │
│                                                                 │
│  Only 0.1% of parameters are trained → fast + memory efficient! │
└─────────────────────────────────────────────────────────────────┘
```

**Key parameters:**
- `r=16` — LoRA rank (size of adapter matrices)
- `lora_alpha=32` — scaling factor
- `4-bit NF4` — quantization type (best for QLoRA)
- Target modules: `q_proj`, `v_proj`, `k_proj`, `o_proj`

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Inspect Training Data**</span>

</div>

In [2]:
# Load our Week 6 training data
with open("../data/processed/train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open("../data/processed/validation.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)

print(f"📊 Training dataset stats:")
print(f"   Train      : {len(train_data)} examples")
print(f"   Validation : {len(val_data)} examples")

print(f"\n📋 Sample training example (Alpaca format):")
sample = train_data[0]
print(f"\nINSTRUCTION:")
print(f"  {sample['instruction']}")
print(f"\nINPUT (first 200 chars):")
print(f"  {sample['input'][:200]}...")
print(f"\nOUTPUT:")
print(f"  {sample['output']}")

# Measure average lengths
input_lens  = [len(ex['input'])  for ex in train_data]
output_lens = [len(ex['output']) for ex in train_data]

print(f"\n📐 Average input length  : {np.mean(input_lens):.0f} chars")
print(f"📐 Average output length : {np.mean(output_lens):.0f} chars")

📊 Training dataset stats:
   Train      : 799 examples
   Validation : 100 examples

📋 Sample training example (Alpaca format):

INSTRUCTION:
  You are a legal expert. Answer the following question based ONLY on the provided legal document context. Be precise and faithful to the source.

INPUT (first 200 chars):
  Context: . (B) Failure to accept offer.--If the contract holder does not accept the offer under paragraph (1) or if an agreement is not negotiated under paragraph (2)(D) within the time period describ...

OUTPUT:
  The contracts shall remain in effect and no further actions shall be taken pursuant to this Act.

📐 Average input length  : 419 chars
📐 Average output length : 115 chars


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Generate the Colab Fine-tuning Notebook**</span>

</div>

This cell generates `colab_week7_finetuning.ipynb` — a ready-to-run Google Colab notebook.

After running this cell:
1. Upload `colab_week7_finetuning.ipynb` to **Google Colab**
2. Upload `data/processed/train.json` and `validation.json` to Colab
3. Set your **HuggingFace token** in the notebook
4. Run all cells (~2 hours on T4 GPU)
5. Your fine-tuned model will be at: `your-hf-username/legal-mistral-7b`

In [4]:
colab_notebook = {
 "nbformat": 4,
 "nbformat_minor": 0,
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "name": "python3"},
  "language_info": {"name": "python"},
  "accelerator": "GPU",
  "colab": {"provenance": [], "gpuType": "T4"}
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# 🚀 Week 7: QLoRA Fine-tuning Mistral-7B\n",
              "**Run this on Google Colab with T4 GPU (free tier)**\n\n",
              "Runtime → Change runtime type → T4 GPU"]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": ["# ⚙️ STEP 1: Install dependencies (~3 minutes)\n",
              "!pip install -q transformers==4.44.0 peft==0.12.0 trl==0.10.1 \\\n",
              "    bitsandbytes==0.43.3 accelerate==0.34.2 datasets==3.0.0 \\\n",
              "    huggingface_hub sentencepiece\n",
              "print('✅ Dependencies installed!')"],
   "outputs": [],
   "execution_count": None
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": ["# 🔑 STEP 2: Login to HuggingFace\n",
              "from huggingface_hub import login\n",
              "\n",
              "HF_TOKEN = \"hf_YOUR_TOKEN_HERE\"  # paste your HuggingFace token\n",
              "HF_USERNAME = \"your-username\"     # your HuggingFace username\n",
              "MODEL_OUTPUT_NAME = \"legal-mistral-7b\"  # name for your fine-tuned model\n",
              "\n",
              "login(token=HF_TOKEN)\n",
              "print(f'✅ Logged in! Model will be saved to: {HF_USERNAME}/{MODEL_OUTPUT_NAME}')"],
   "outputs": [],
   "execution_count": None
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": ["# 📂 STEP 3: Upload and load training data\n",
              "# Upload train.json and validation.json using the Files panel (left sidebar)\n",
              "import json\n",
              "\n",
              "with open('train.json', 'r') as f:\n",
              "    train_data = json.load(f)\n",
              "with open('validation.json', 'r') as f:\n",
              "    val_data = json.load(f)\n",
              "\n",
              "print(f'✅ Train: {len(train_data)} examples')\n",
              "print(f'✅ Val  : {len(val_data)} examples')"],
   "outputs": [],
   "execution_count": None
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": ["# 🗂️ STEP 4: Convert to HuggingFace Dataset format\n",
              "from datasets import Dataset\n",
              "\n",
              "def format_alpaca(example):\n",
              "    return {\n",
              "        'text': f\"### Instruction:\\n{example['instruction']}\\n\\n\"",
              "                f\"### Input:\\n{example['input']}\\n\\n\"",
              "                f\"### Response:\\n{example['output']}\"\n",
              "    }\n",
              "\n",
              "train_dataset = Dataset.from_list([format_alpaca(ex) for ex in train_data])\n",
              "val_dataset   = Dataset.from_list([format_alpaca(ex) for ex in val_data])\n",
              "\n",
              "print(f'✅ Train dataset: {len(train_dataset)} rows')\n",
              "print(f'✅ Val dataset  : {len(val_dataset)} rows')\n",
              "print(f'\\n📋 Sample formatted text:')\n",
              "print(train_dataset[0]['text'][:400])"],
   "outputs": [],
   "execution_count": None
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": ["# 🤖 STEP 5: Load Mistral-7B in 4-bit (QLoRA)\n",
              "import torch\n",
              "from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n",
              "\n",
              "BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'\n",
              "\n",
              "# 4-bit quantization config\n",
              "bnb_config = BitsAndBytesConfig(\n",
              "    load_in_4bit=True,\n",
              "    bnb_4bit_quant_type='nf4',\n",
              "    bnb_4bit_compute_dtype=torch.float16,\n",
              "    bnb_4bit_use_double_quant=True\n",
              ")\n",
              "\n",
              "print('⏳ Loading Mistral-7B in 4-bit... (~5 minutes)')\n",
              "\n",
              "tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)\n",
              "tokenizer.pad_token = tokenizer.eos_token\n",
              "tokenizer.padding_side = 'right'\n",
              "\n",
              "model = AutoModelForCausalLM.from_pretrained(\n",
              "    BASE_MODEL,\n",
              "    quantization_config=bnb_config,\n",
              "    device_map='auto',\n",
              "    trust_remote_code=True\n",
              ")\n",
              "model.config.use_cache = False\n",
              "model.config.pretraining_tp = 1\n",
              "\n",
              "print(f'✅ Model loaded! Parameters: {model.num_parameters()/1e9:.1f}B')"],
   "outputs": [],
   "execution_count":None
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": ["# 🔧 STEP 6: Configure LoRA adapters\n",
              "from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n",
              "\n",
              "model = prepare_model_for_kbit_training(model)\n",
              "\n",
              "lora_config = LoraConfig(\n",
              "    r=16,                    # LoRA rank\n",
              "    lora_alpha=32,           # scaling factor\n",
              "    target_modules=[         # which layers to adapt\n",
              "        'q_proj', 'v_proj',\n",
              "        'k_proj', 'o_proj',\n",
              "        'gate_proj', 'up_proj', 'down_proj'\n",
              "    ],\n",
              "    lora_dropout=0.05,\n",
              "    bias='none',\n",
              "    task_type='CAUSAL_LM'\n",
              ")\n",
              "\n",
              "model = get_peft_model(model, lora_config)\n",
              "\n",
              "trainable, total = model.print_trainable_parameters()\n",
              "print(f'✅ LoRA configured!')"],
   "outputs": [],
   "execution_count": None
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": ["# 🏋️ STEP 7: Fine-tune with SFTTrainer (~2 hours on T4)\n",
              "from transformers import TrainingArguments\n",
              "from trl import SFTTrainer\n",
              "\n",
              "training_args = TrainingArguments(\n",
              "    output_dir='./results',\n",
              "    num_train_epochs=3,\n",
              "    per_device_train_batch_size=4,\n",
              "    gradient_accumulation_steps=4,\n",
              "    optim='paged_adamw_32bit',\n",
              "    save_steps=200,\n",
              "    logging_steps=50,\n",
              "    learning_rate=2e-4,\n",
              "    weight_decay=0.001,\n",
              "    fp16=True,\n",
              "    bf16=False,\n",
              "    max_grad_norm=0.3,\n",
              "    max_steps=-1,\n",
              "    warmup_ratio=0.03,\n",
              "    group_by_length=True,\n",
              "    lr_scheduler_type='cosine',\n",
              "    report_to='none',\n",
              "    evaluation_strategy='steps',\n",
              "    eval_steps=200,\n",
              "    load_best_model_at_end=True\n",
              ")\n",
              "\n",
              "trainer = SFTTrainer(\n",
              "    model=model,\n",
              "    train_dataset=train_dataset,\n",
              "    eval_dataset=val_dataset,\n",
              "    peft_config=lora_config,\n",
              "    dataset_text_field='text',\n",
              "    max_seq_length=512,\n",
              "    tokenizer=tokenizer,\n",
              "    args=training_args,\n",
              "    packing=False\n",
              ")\n",
              "\n",
              "print('⏳ Starting fine-tuning... (~2 hours on T4 GPU)')\n",
              "print('☕ Take a long break!\\n')\n",
              "trainer.train()\n",
              "print('\\n✅ Fine-tuning complete!')"],
   "outputs": [],
   "execution_count":None
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": ["# 💾 STEP 8: Save and push to HuggingFace Hub\n",
              "repo_id = f'{HF_USERNAME}/{MODEL_OUTPUT_NAME}'\n",
              "\n",
              "print(f'⏳ Pushing model to {repo_id}...')\n",
              "trainer.model.push_to_hub(repo_id, use_auth_token=True)\n",
              "tokenizer.push_to_hub(repo_id, use_auth_token=True)\n",
              "\n",
              "print(f'\\n✅ Model uploaded to HuggingFace!')\n",
              "print(f'🔗 https://huggingface.co/{repo_id}')\n",
              "print(f'\\n📝 Copy this model ID for your local evaluation notebook:')\n",
              "print(f'   MODEL_ID = \"{repo_id}\"')"],
   "outputs": [],
   "execution_count": None
  }
 ]
}

with open("colab_week7_finetuning.ipynb", "w", encoding="utf-8") as f:
    json.dump(colab_notebook, f, indent=1)

print("✅ Generated: colab_week7_finetuning.ipynb")
print("\n📋 Next steps:")
print("   1. Go to https://colab.research.google.com")
print("   2. File → Upload notebook → colab_week7_finetuning.ipynb")
print("   3. Runtime → Change runtime type → T4 GPU")
print("   4. Upload train.json + validation.json (Files panel, left sidebar)")
print("   5. Set your HuggingFace token in Step 2")
print("   6. Runtime → Run all  (~2 hours)")
print("\n🔑 Get your HuggingFace token at:")
print("   https://huggingface.co/settings/tokens")
print("   (Create a token with WRITE access)")

✅ Generated: colab_week7_finetuning.ipynb

📋 Next steps:
   1. Go to https://colab.research.google.com
   2. File → Upload notebook → colab_week7_finetuning.ipynb
   3. Runtime → Change runtime type → T4 GPU
   4. Upload train.json + validation.json (Files panel, left sidebar)
   5. Set your HuggingFace token in Step 2
   6. Runtime → Run all  (~2 hours)

🔑 Get your HuggingFace token at:
   https://huggingface.co/settings/tokens
   (Create a token with WRITE access)


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Setup Groq & Rebuild RAG Pipeline**</span>

</div>

In [ ]:
# Paste your HuggingFace model ID after Colab training
MODEL_ID = "your-username/legal-mistral-7b"  # ← change this!
HF_TOKEN = "hf_YOUR_TOKEN_HERE"              # ← your HuggingFace token

# Setup Groq (for RAGAS evaluation judge)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key="GROQ_API_KEY"
)

# Setup RAGAS metrics
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)
faithfulness.llm = ragas_llm
answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_embeddings
context_precision.llm = ragas_llm

print("✅ Groq + RAGAS ready!")
print(f"✅ Fine-tuned model: {MODEL_ID}")

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Rebuild RAG Pipeline (Best config from Week 5)**</span>

</div>

In [ ]:
# Load data
with open("../data/raw/legal_documents.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

with open("../data/processed/qa_pairs.json", "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

eval_sample = qa_pairs[:50]
print(f"✅ Loaded {len(documents)} documents, {len(qa_pairs)} QA pairs")

# Build chunks (chunk256 — best from Week 3/4)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=256, chunk_overlap=25,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = []
for doc in documents:
    for i, chunk_text in enumerate(splitter.split_text(doc["text"])):
        chunks.append({"text": chunk_text, "doc_id": doc["doc_id"]})

print(f"✅ Created {len(chunks)} chunks")

# Build FAISS + BM25
print("⏳ Building search indexes...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(
    [c["text"] for c in chunks],
    batch_size=64, show_progress_bar=True, convert_to_numpy=True
)
faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings.astype(np.float32))

def tokenize(text):
    return re.findall(r'\w+', text.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("✅ FAISS, BM25, and reranker ready!")

In [ ]:
# Hybrid search + reranking (best pipeline from Week 4)
def hybrid_search_with_reranking(query, top_k=3, candidate_k=10):
    query_vec = embed_model.encode([query], convert_to_numpy=True).astype(np.float32)
    _, faiss_idx = faiss_index.search(query_vec, 50)
    faiss_rank = {idx: rank for rank, idx in enumerate(faiss_idx[0])}

    bm25_scores = bm25.get_scores(tokenize(query))
    bm25_idx = np.argsort(bm25_scores)[::-1][:50]
    bm25_rank = {idx: rank for rank, idx in enumerate(bm25_idx)}

    all_idx = set(faiss_rank) | set(bm25_rank)
    rrf = {i: 1/(60+faiss_rank.get(i,1000)) + 1/(60+bm25_rank.get(i,1000)) for i in all_idx}
    top = sorted(rrf, key=rrf.get, reverse=True)[:candidate_k]

    candidates = [chunks[i]["text"] for i in top]
    scores = reranker.predict([[query, c] for c in candidates])
    ranked = sorted(zip(scores, candidates), reverse=True)
    return [c for _, c in ranked[:top_k]]

print("✅ Hybrid search + reranking ready!")

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Generate Answers: Baseline vs Fine-tuned**</span>

</div>

We compare two answer generators:
- **Baseline**: Groq `llama-3.1-8b-instant` (no fine-tuning)
- **Fine-tuned**: Your `legal-mistral-7b` via HuggingFace Inference API

In [ ]:
# Baseline answer generator (Groq Llama)
def generate_answer_baseline(question, contexts):
    context_str = "\n\n".join(contexts)
    prompt = f"""You are a legal expert. Answer the question based ONLY on the provided context.
Be precise and faithful to the source.

Context:
{context_str}

Question: {question}

Answer:"""
    return llm.invoke(prompt).content.strip()


# Fine-tuned answer generator (HuggingFace Inference API)
hf_client = InferenceClient(model=MODEL_ID, token=HF_TOKEN)

def generate_answer_finetuned(question, contexts):
    context_str = "\n\n".join(contexts)
    instruction = "You are a legal expert. Answer the following question based ONLY on the provided legal document context. Be precise and faithful to the source."
    input_text  = f"Context: {context_str}\n\nQuestion: {question}"

    prompt = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""
    response = hf_client.text_generation(
        prompt,
        max_new_tokens=200,
        temperature=0.1,
        do_sample=True,
        stop_sequences=["### ", "\n\n\n"]
    )
    return response.strip()


# Quick sanity test
print("🔍 Testing both generators...")
test_q = eval_sample[0]["question"]
test_ctx = hybrid_search_with_reranking(test_q)

print(f"\n❓ Question: {test_q}")
print(f"\n📄 Baseline answer:")
print(f"   {generate_answer_baseline(test_q, test_ctx)[:300]}")
print(f"\n🎯 Fine-tuned answer:")
print(f"   {generate_answer_finetuned(test_q, test_ctx)[:300]}")

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**RAGAS Evaluation: Baseline vs Fine-tuned**</span>

</div>

⏳ Running RAGAS on 50 examples for both models (~40 minutes total)

In [ ]:
def run_ragas(answer_fn, run_name):
    print(f"\n⏳ Evaluating: {run_name}")
    eval_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

    for i, qa in enumerate(eval_sample):
        try:
            contexts = hybrid_search_with_reranking(qa["question"])
            answer   = answer_fn(qa["question"], contexts)
            eval_data["question"].append(qa["question"])
            eval_data["answer"].append(answer)
            eval_data["contexts"].append(contexts)
            eval_data["ground_truth"].append(qa["answer"])
        except Exception as e:
            print(f"  ⚠️ Failed example {i}: {e}")
        time.sleep(0.3)

    dataset = Dataset.from_dict(eval_data)
    results = evaluate(dataset=dataset, metrics=[faithfulness, answer_relevancy, context_precision])
    df = results.to_pandas()

    f_score = df['faithfulness'].dropna().mean()
    r_score = df['answer_relevancy'].dropna().mean()
    p_score = df['context_precision'].dropna().mean()

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("technique", run_name)
        mlflow.log_param("chunk_size", 256)
        mlflow.log_param("embedding_model", "all-MiniLM-L6-v2")
        mlflow.log_param("search_type", "hybrid+reranking")
        mlflow.log_param("answer_model", run_name)
        mlflow.log_metric("faithfulness", f_score)
        mlflow.log_metric("answer_relevancy", r_score)
        mlflow.log_metric("context_precision", p_score)

    print(f"✅ {run_name} Results:")
    print(f"   Faithfulness      : {f_score:.4f}")
    print(f"   Answer Relevancy  : {r_score:.4f}")
    print(f"   Context Precision : {p_score:.4f}")
    return f_score, r_score, p_score


print("⏳ Running evaluations... (~40 minutes total)")
print("☕ Take a break!\n")

baseline_scores   = run_ragas(generate_answer_baseline,   "Baseline_Llama")
finetuned_scores  = run_ragas(generate_answer_finetuned,  "Finetuned_Mistral7B")

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Week 7 Summary**</span>

</div>

In [ ]:
b_f, b_r, b_p = baseline_scores
ft_f, ft_r, ft_p = finetuned_scores

print("=" * 65)
print("🎉 WEEK 7 - QLoRA FINE-TUNING COMPLETE!")
print("=" * 65)

print(f"""
📊 Results: Baseline vs Fine-tuned Mistral-7B:

┌─────────────────────────────┬──────────┬──────────┬──────────┐
│ Model                       │ Faith.   │ Relev.   │ Precis.  │
├─────────────────────────────┼──────────┼──────────┼──────────┤
│ Baseline (Llama-8b)         │ {b_f:.4f}   │ {b_r:.4f}   │ {b_p:.4f}   │
│ Fine-tuned (Mistral-7B)     │ {ft_f:.4f}   │ {ft_r:.4f}   │ {ft_p:.4f}   │
├─────────────────────────────┼──────────┼──────────┼──────────┤
│ Improvement                 │ {ft_f-b_f:+.4f}   │ {ft_r-b_r:+.4f}   │ {ft_p-b_p:+.4f}   │
└─────────────────────────────┴──────────┴──────────┴──────────┘
""")

print("📊 Full Progress — All Experiments:")
print("""
┌────────────────────────────┬──────────┬──────────┬──────────┐
│ Experiment                 │ Faith.   │ Relev.   │ Precis.  │
├────────────────────────────┼──────────┼──────────┼──────────┤
│ Baseline (512+MiniLM)      │ 0.5750   │ 0.6105   │ 0.5784   │
│ Week3: chunk256+MiniLM     │ 0.6775   │ 0.5741   │ 0.4048   │
│ Week4: Hybrid+Reranking    │ 0.6104   │ 0.5889   │ 0.7692 🚀│
│ Week5: QueryExpansion      │ 0.6780 🚀│ 0.5723   │ 0.6364   │
│ Week7: Fine-tuned Mistral  │ {ft_f:.4f} {'🚀' if ft_f > 0.7 else '  '}│ {ft_r:.4f}   │ {ft_p:.4f}   │
└────────────────────────────┴──────────┴──────────┴──────────┘
""")

print(f"""
🔜 Next — Week 8: RAG Evaluation Dashboard
   → Build a Streamlit app to visualize all results
   → Interactive query interface with the full pipeline
   → Compare models side by side
   → Deploy as a demo!
""")
print("=" * 65)